In [0]:
-- ==============================================================================
-- Gold Layer: Dimensional Modeling (Star Schema)
-- Capa Oro: Modelado dimensional (esquema en estrella)
-- Source: ecommerce_silver.cleansed_sales
-- Target Schema: ecommerce_gold
-- ==============================================================================

-- 1. Create Gold Schema
CREATE SCHEMA IF NOT EXISTS ecommerce_gold;

In [0]:
-- ------------------------------------------------------------------------------
-- 2. DIMENSION: dim_customers
-- ------------------------------------------------------------------------------
CREATE OR REPLACE TABLE ecommerce_gold.dim_customers
USING DELTA
AS
SELECT DISTINCT
    customer_key,
    customer_id,
    CURRENT_TIMESTAMP() AS created_at
FROM ecommerce_silver.cleansed_sales;


In [0]:
-- ------------------------------------------------------------------------------
-- 3. DIMENSION: dim_products
-- ------------------------------------------------------------------------------
CREATE OR REPLACE TABLE ecommerce_gold.dim_products
USING DELTA
AS
SELECT DISTINCT
    MD5(product_category) AS product_key,
    product_category,
    CURRENT_TIMESTAMP() AS created_at
FROM ecommerce_silver.cleansed_sales;


In [0]:
-- ------------------------------------------------------------------------------
-- 4. DIMENSION: dim_date
-- Dynamic Date Table generated from Silver order_date range
-- Tabla de fechas dinámica generada a partir del rango de order_date de Silver
-- ------------------------------------------------------------------------------
CREATE OR REPLACE TABLE ecommerce_gold.dim_date
USING DELTA
AS
WITH date_range AS (
    SELECT 
        MIN(order_date) AS min_date,
        MAX(order_date) AS max_date
    FROM ecommerce_silver.cleansed_sales
),
dates AS (
    SELECT 
        EXPLODE(SEQUENCE(min_date, max_date, INTERVAL 1 DAY)) AS full_date
    FROM date_range
)
SELECT 
    CAST(DATE_FORMAT(full_date, 'yyyyMMdd') AS INT) AS date_key,
    full_date                                       AS order_date,
    YEAR(full_date)                                 AS year,
    QUARTER(full_date)                              AS quarter,
    MONTH(full_date)                                AS month,
    DATE_FORMAT(full_date, 'MMMM')                  AS month_name,
    DAY(full_date)                                  AS day,
    DAYOFWEEK(full_date)                            AS day_of_week,
    DATE_FORMAT(full_date, 'EEEE')                  AS day_name
FROM dates;

-- If we wanted to apply this to a larger-scale venture involving a large volume of data, the next step in maturity would be to implement a "Buffer Range" (extending the date range from January 1st of the first year through December 31st of the following year). Since a 10-year date table occupies only a few kilobytes of memory in Spark SQL, we ensure full coverage in BI dashboards without any performance impact

In [0]:
-- ------------------------------------------------------------------------------
-- 5. FACT TABLE: fact_sales
-- ------------------------------------------------------------------------------
CREATE OR REPLACE TABLE ecommerce_gold.fact_sales
USING DELTA
AS
SELECT 
    s.order_id,
    
    -- Dimensional Foreign Keys
    -- Claves foráneas dimensionales
    -- =====================================
    s.customer_key,
    MD5(s.product_category)                              AS product_key,
    CAST(DATE_FORMAT(s.order_date, 'yyyyMMdd') AS INT)   AS date_key,
    
    -- Degenerate Dimensions / Categoricals
    -- =====================================
    s.region,
    s.payment_method,
    
    -- Measures / Metrics
    -- =====================================
    s.quantity,
    s.unit_price,
    s.discount,
    s.revenue,
    s.delivery_days,
    s.customer_rating,
    
    -- Metadata
    -- =====================================
    CURRENT_TIMESTAMP() AS created_at

FROM ecommerce_silver.cleansed_sales s;

In [0]:
SELECT * 
FROM ecommerce_gold.fact_sales 
LIMIT 10;